In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: /home/scinolimits/robotics/image_processing_and_computer_vision/shortcut-learning-analysis
src exists: True


In [4]:
import torch
from src.model import FashionCNN

model = FashionCNN()

# Batch of 32 RGB images
dummy_input = torch.randn(32, 3, 28, 28)

output = model(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

Input shape: torch.Size([32, 3, 28, 28])
Output shape: torch.Size([32, 10])
Total parameters: 422218


In [9]:
import numpy as np
from pathlib import Path
from torchvision.datasets import FashionMNIST

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

output_dir = (
    PROJECT_ROOT
    / "data"
    / "FashionMNIST"
    / "processed"
    / "controlled_intervention"
)

print("Dataset directory:", output_dir)
print("Directory exists:", output_dir.exists())

images = np.load(output_dir / "intervention_images.npy")
labels = np.load(output_dir / "intervention_labels.npy")

# Load original grayscale test images
dataset = FashionMNIST(
    root="../data",
    train=False,
    download=False
)

original_grayscale = np.array(
    [np.array(dataset[i][0], dtype=np.uint8) for i in range(len(dataset))]
)

original_labels = np.array(
    [dataset[i][1] for i in range(len(dataset))]
)

print("Images:", images.shape)
print("Labels:", labels.shape)

# Build foreground mask from original grayscale images
foreground_mask = original_grayscale >= 10
foreground_mask = np.repeat(
    foreground_mask[:, :, :, np.newaxis],
    3,
    axis=3
)

# Verify non-background pixels remain identical
original = images[:, 0]

for cue_class in range(1, 10):
    variation = images[:, cue_class]

    unchanged = np.array_equal(
        original[foreground_mask],
        variation[foreground_mask]
    )

    print(
        f"Cue {cue_class}: "
        f"non-background pixels unchanged = {unchanged}"
    )

# Verify labels
assert images.shape == (10000, 10, 28, 28, 3)
assert labels.shape == (10000,)
assert np.array_equal(labels, original_labels)

print("\nIntervention dataset validation passed.")

Dataset directory: /home/scinolimits/robotics/image_processing_and_computer_vision/shortcut-learning-analysis/data/FashionMNIST/processed/controlled_intervention
Directory exists: True
Images: (10000, 10, 28, 28, 3)
Labels: (10000,)
Cue 1: non-background pixels unchanged = True
Cue 2: non-background pixels unchanged = True
Cue 3: non-background pixels unchanged = True
Cue 4: non-background pixels unchanged = True
Cue 5: non-background pixels unchanged = True
Cue 6: non-background pixels unchanged = True
Cue 7: non-background pixels unchanged = True
Cue 8: non-background pixels unchanged = True
Cue 9: non-background pixels unchanged = True

Intervention dataset validation passed.
